# QLoRA fine-tuning: ticket-router (Qwen2.5-1.5B-Instruct)

Fine-tunes the open-weights model set in `train/config.yaml` (`base_model`, default
`Qwen/Qwen2.5-1.5B-Instruct`) on the ticket-routing SFT data from step-06, so it can be
self-hosted, quantized, and served in Docker in later steps (09-12). **Not** a hosted-API
fine-tune -- see the project's `README.md` "Next steps" section for why.

**Runs on Google Colab only (free T4 GPU runtime). No training happens on the local machine.**

## Expected runtime (free T4, approximate -- varies with Colab's GPU allocation/throttling)
- Smoke test (~100 steps): **~5-10 minutes** (now also prints a checkpoint disk-size check)
- Full run (3 epochs, ~2,450 steps at effective batch size 16, eval/save every 250 steps
  against a 400-example subsample): **~45-90 minutes**
- Final best-checkpoint selection (`save_total_limit` is null -- EVERY checkpoint from the
  full run survives, ~10 of them at save_steps=250 -- each gets one full-validation pass,
  1,635 examples): **~+15-30 minutes**

## Expected VRAM (free T4 has 16 GB total)
- **~6-10 GB** during training (4-bit NF4 base model + LoRA adapters + activations with
  gradient checkpointing + paged 8-bit optimizer states) -- comfortable headroom on a T4.

## Expected disk (Colab-local or Google Drive, wherever `REPO_ROOT` points)
- LoRA adapter checkpoints only (not full model weights) -- a few tens of MB each. The
  smoke-test cell (step 11) measures one real checkpoint's size and prints the projected
  total for all ~10 full-run checkpoints (`save_total_limit` is null, so none get deleted),
  compared against free disk space at `REPO_ROOT`. If it doesn't fit, the cell prints a
  warning and stops short of recommending a fix -- decide with the user before changing
  `save_total_limit`, rather than silently shrinking it.

## Cell order -- run top to bottom, in this exact order
1. Check GPU (must show a T4 or better -- stop if none) -- also decides bf16 vs. fp16 for this
   session's GPU, printed here before anything else runs
2. Install dependencies
3. Get the repo (git clone for code + Google Drive/upload for the gitignored data files)
4. Load `train/config.yaml`
5. **Check a Hugging Face token with WRITE access is present** -- before any training runs,
   not after a ~45-90 minute run is done and the push step fails
6. Load tokenizer + SFT datasets (full validation set, and a fixed seeded subsample used only
   for speed during mid-training evaluation -- see `train/config.yaml`)
7. **Verify completion-only masking** (prints one full example's per-token labels)
8. **Verify padding is dynamic per-batch, not fixed to `max_seq_len`**
9. Load base model in 4-bit (QLoRA) + attach LoRA
10. Build the trainer factory from `train/config.yaml` (nothing hardcoded here)
11. **Smoke test** (~100 steps) -- loss curve + 5 generated JSON outputs + a checkpoint
    disk-size projection for the full run
12. --- STOP AND CONFIRM before continuing (read the smoke-test results first, INCLUDING the
    disk projection) ---
13. **Full training run** (mid-training eval/loss-curve uses the SUBSAMPLE, for speed;
    `save_total_limit` is null, so every checkpoint is kept)
14. **Select the best checkpoint using the FULL validation set** -- the subsample used during
    training is for speed only and is never used to pick the final checkpoint; every
    checkpoint saved during the run is evaluated here, since none were deleted
15. Save the adapter + push it to the Hugging Face Hub
16. **Post-train preview**: generate + print alongside gold labels on 20 val examples

> **A note on what was and wasn't verified before this notebook was written**: the
> completion-only-masking mechanism (cell 7) and the dynamic-padding behavior (cell 8) were
> verified by hand, token-by-token, against Qwen2.5's real tokenizer before this notebook was
> written -- see `PROGRESS.md`'s step-07 entry for the full worked example. The end-to-end
> QLoRA training loop itself (bitsandbytes 4-bit loading, the actual GPU training step, and the
> new full-validation checkpoint-selection cell) could **not** be executed locally (no GPU
> available, and a local disk-space constraint blocked even a CPU-only `trl`/`torch` install for
> further testing) -- it's written to the current standard `transformers`/`trl`/`peft` API, but
> you should expect to debug minor version-API mismatches on first run. Please report back what
> you see so this can be corrected if needed.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Runtime menu -> Change runtime type -> T4 GPU -> Save, then re-run from the top."
    )

# bf16 needs Ampere-or-newer (CUDA compute capability major >= 8). The free Colab tier usually
# allocates a T4, which is Turing (compute capability 7.5) and does NOT support bf16 -- decide
# this at runtime from the actual allocated GPU rather than hardcoding bfloat16 in config.yaml,
# which would be silently wrong on a T4. COMPUTE_DTYPE/USE_BF16 are used below by both the QLoRA
# BitsAndBytesConfig (cell 9) and the trainer's mixed-precision flag (cell 10).
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print(f"GPU detected: {gpu_name} (compute capability {major}.{minor})")
print(f"bf16 supported: {USE_BF16} -> using compute dtype: {COMPUTE_DTYPE}")

## 2. Install dependencies

Training-specific packages, installed only inside this Colab runtime -- not part of the
project's own `requirements.txt` (that file is for the local data/harness pipeline, which
never needs a GPU training stack).

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" datasets pyyaml huggingface_hub

## 3. Get the repo

`data/processed/` (the parquet splits and the SFT `.jsonl` files from step-06) is **gitignored** -- a plain `git clone` will get you the notebook, `train/config.yaml`, and the `src/` package, but **not** the actual training data. Pick ONE of the two options below.

**Option A (recommended if you've pushed this repo to GitHub):** clone for the code, then upload just the two `.jsonl` files (small enough for a direct Colab upload).

**Option B (no GitHub remote needed):** sync your entire local repo folder -- gitignored files included -- to Google Drive, then mount Drive here.

In [ ]:
import os

# --- Option A: git clone (code) + direct upload (data) ---------------------------------
# !git clone https://github.com/<your-username>/ticket-router.git
# REPO_ROOT = "/content/ticket-router"
# Then use the Colab file-browser (left sidebar) to upload train.jsonl/val.jsonl into
# REPO_ROOT + "/data/processed/sft/", or use the Files > Upload widget.

# --- Option B: Google Drive (default -- edit DRIVE_REPO_PATH to match your setup) ------
from google.colab import drive
drive.mount('/content/drive')
DRIVE_REPO_PATH = "/content/drive/MyDrive/ticket-router"  # <-- EDIT to where you placed the repo
REPO_ROOT = DRIVE_REPO_PATH

assert os.path.isdir(REPO_ROOT), f"{REPO_ROOT} not found -- upload/sync the repo there, or use Option A above."
assert os.path.exists(f"{REPO_ROOT}/train/config.yaml"), "train/config.yaml not found under REPO_ROOT."
assert os.path.exists(f"{REPO_ROOT}/data/processed/sft/train.jsonl"), (
    "data/processed/sft/train.jsonl not found. This directory is gitignored, so a plain "
    "git clone alone will NOT include it -- run src/ticket_router/make_sft_dataset.py "
    "locally first, then upload/sync data/processed/sft/*.jsonl separately."
)
print("REPO_ROOT OK:", REPO_ROOT)

## 4. Load `train/config.yaml`

Every hyperparameter used below comes from this file -- nothing is hardcoded in this notebook.

In [ ]:
import sys, yaml

sys.path.insert(0, f"{REPO_ROOT}/src")  # so `from ticket_router...` works later (schema reuse)

with open(f"{REPO_ROOT}/train/config.yaml") as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, sort_keys=False))

## 5. Check Hugging Face token (write access)

Training pushes the adapter to `hub_model_id` from `train/config.yaml`
(`Omar-Mohsen22/ticket-router-qwen2.5-1.5b-qlora`) at the very end. Checking this **now**, before
the smoke test or the full run, means a missing/insufficient token fails in seconds -- not after
a ~45-90 minute training run, at the push step.

In [ ]:
from huggingface_hub import HfApi, get_token
from huggingface_hub.utils import HfHubHTTPError

token = get_token()  # HF_TOKEN env var, a Colab "HF_TOKEN" secret, or a prior login() call
if not token:
    raise RuntimeError(
        "No Hugging Face token found. This notebook pushes the trained adapter to "
        f"'{config['hub_model_id']}' (train/config.yaml's hub_model_id) and needs a WRITE-access "
        "token before training starts, not after. In Colab: add a secret named HF_TOKEN (key "
        "icon, left sidebar) holding a token with 'Write' access "
        "(https://huggingface.co/settings/tokens), or run `from huggingface_hub import login; "
        "login()` in a cell above and re-run this cell."
    )

api = HfApi(token=token)
try:
    who = api.whoami()
    # Token permission fields vary in shape by token type (classic vs. fine-grained), so the
    # reliable check is a real write probe: create the target repo now if it doesn't exist yet
    # (harmless and idempotent, exist_ok=True) rather than parsing whoami()'s scope structure.
    api.create_repo(config["hub_model_id"], private=config["hub_private"], exist_ok=True)
except HfHubHTTPError as e:
    raise RuntimeError(
        f"Hugging Face token found but does not have WRITE access to "
        f"'{config['hub_model_id']}'. Generate a token with 'Write' access at "
        f"https://huggingface.co/settings/tokens, then set it as the HF_TOKEN Colab secret or "
        f"call login() again.\nOriginal error: {e}"
    ) from e

print(f"HF token OK: logged in as '{who['name']}', write access to '{config['hub_model_id']}' confirmed.")

## 6. Load tokenizer + SFT datasets

Loads the full validation set (used only once, for final best-checkpoint selection in step 14)
and a fixed, seeded subsample of it (`eval_subsample_size`/`eval_subsample_seed` in
`train/config.yaml`) used for speed during mid-training evaluation in steps 11 and 13.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(config["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_path = f"{REPO_ROOT}/{config['train_file']}"
val_path = f"{REPO_ROOT}/{config['val_file']}"

raw_datasets = load_dataset("json", data_files={"train": train_path, "val": val_path})
print(raw_datasets)
print("\nOne raw example's messages:")
print(raw_datasets["train"][0]["messages"])

# Fixed, seeded subsample of the full validation set -- SPEED ONLY, used for mid-training
# evaluation (steps 11 and 13's loss curves). The FULL validation set (raw_datasets["val"]) is
# kept untouched and is used exactly once, for final best-checkpoint selection in step 14. See
# train/config.yaml's "Evaluation & checkpointing" comments for why these are kept separate.
eval_subsample = raw_datasets["val"].shuffle(seed=config["eval_subsample_seed"]).select(
    range(min(config["eval_subsample_size"], len(raw_datasets["val"])))
)
print(
    f"\nFull validation set: {len(raw_datasets['val'])} examples (final checkpoint selection only)."
    f"\nMid-training eval subsample: {len(eval_subsample)} examples "
    f"(seed={config['eval_subsample_seed']}, speed only -- NOT the metric used to pick the final checkpoint)."
)

## 7. Verify completion-only masking

The loss must be computed **only** on the assistant's JSON response -- system and user tokens
are masked out (label = `-100`, PyTorch's cross-entropy `ignore_index`, so they contribute
nothing to the loss or its gradient). Otherwise most of the model's fine-tuning capacity goes
into learning to *write* support tickets (reproducing the user turn) instead of learning to
*classify* them (producing the assistant turn).

**Mechanism: `trl.DataCollatorForCompletionOnlyLM`**, given the exact token sequence for
Qwen's chat-template assistant marker (`response_template` from `train/config.yaml`,
`"<|im_start|>assistant\n"`).

Verified by hand before this notebook was written (see `PROGRESS.md`'s step-07 entry for the
full worked example): for Qwen2.5's tokenizer, `<|im_start|>` and `<|im_end|>` are **protected
special tokens**, so `response_template` tokenizes identically whether it's standalone or
embedded in a full formatted sequence -- no silent boundary mismatch (the classic failure mode
with this collator on tokenizers that don't protect these markers). The cell below reproduces
that exact check live, using the actual `trl` collator, and prints one full example's
per-token labels so you can see the prompt tokens are ignored.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

response_template = config["response_template"]
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)

def tokenize_example(example):
    ids = tokenizer.apply_chat_template(
        example["messages"], tokenize=True, add_generation_prompt=False,
        truncation=True, max_length=config["max_seq_len"],
    )
    return {"input_ids": ids}

tokenized_train = raw_datasets["train"].map(tokenize_example, remove_columns=raw_datasets["train"].column_names)
tokenized_val = raw_datasets["val"].map(tokenize_example, remove_columns=raw_datasets["val"].column_names)

# Sanity-check the collator on ONE real example.
example = tokenized_train[0]
batch = collator([{"input_ids": example["input_ids"]}])
input_ids = batch["input_ids"][0].tolist()
labels = batch["labels"][0].tolist()

n_masked = sum(1 for l in labels if l == -100)
n_loss = len(labels) - n_masked
print(f"Example 0: {len(labels)} tokens total, {n_masked} masked (-100), {n_loss} contribute to loss.\n")

print(f"{'idx':>4} {'token_id':>8}  {'label':>8}  token_text")
for i, (tid, lab) in enumerate(zip(input_ids, labels)):
    tok_str = tokenizer.convert_ids_to_tokens([tid])[0]
    marker = "MASKED" if lab == -100 else "LOSS  "
    print(f"{i:>4} {tid:>8}  {str(lab):>8}  [{marker}] {tok_str!r}")

loss_token_ids = [tid for tid, lab in zip(input_ids, labels) if lab != -100]
print("\nDecoded loss-only tokens (should be exactly the assistant JSON + <|im_end|>):")
print(repr(tokenizer.decode(loss_token_ids)))

assert n_loss > 0, (
    "No unmasked tokens found -- response_template did not match this example. "
    "STOP: do not train until this is fixed, or every example's loss would be masked entirely."
)

## 8. Confirm padding: dynamic per-batch, not fixed to `max_seq_len`

`max_seq_len` in `train/config.yaml` is a **truncation cap**, not a padding target (this was
left open in step-06). Verified by hand before this notebook was written: `tokenizer.pad()`
(which `DataCollatorForCompletionOnlyLM` uses internally) pads each batch to **that batch's
own longest sequence**, not to a fixed length. The cell below reproduces that check against
the actual collator built above, with two differently-sized batches.

In [ ]:
batch_a = [tokenized_train[i] for i in range(4)]
batch_b = [tokenized_train[i] for i in range(4, 4 + 8)]  # a differently-sized batch

padded_a = collator([{"input_ids": ex["input_ids"]} for ex in batch_a])
padded_b = collator([{"input_ids": ex["input_ids"]} for ex in batch_b])

print("batch A native lengths:", [len(ex["input_ids"]) for ex in batch_a])
print("batch A padded shape:  ", tuple(padded_a["input_ids"].shape))
print()
print("batch B native lengths:", [len(ex["input_ids"]) for ex in batch_b])
print("batch B padded shape:  ", tuple(padded_b["input_ids"].shape))
print()
print(f"max_seq_len from config: {config['max_seq_len']}")
print("If padding were fixed to max_seq_len, BOTH batches would show that same second dimension.")
print("Instead each batch is padded independently to its own longest row -- dynamic per-batch padding, confirmed.")

## 9. Load base model in 4-bit (QLoRA) + attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config["quantization"]["load_in_4bit"],
    bnb_4bit_quant_type=config["quantization"]["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # decided at runtime in cell 1 (GPU check), not from config.yaml
    bnb_4bit_use_double_quant=config["quantization"]["bnb_4bit_use_double_quant"],
)

model = AutoModelForCausalLM.from_pretrained(
    config["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=config["gradient_checkpointing"]
)

lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    bias=config["lora"]["bias"],
    task_type=config["lora"]["task_type"],
    target_modules=config["lora"]["target_modules"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 10. Trainer factory

One function, reused for both the smoke test and the full run, so the two runs differ only in
`max_steps`/`num_train_epochs`/`eval_dataset` -- every other hyperparameter comes from
`train/config.yaml`. `load_best_model_at_end` is always `False` here (see `train/config.yaml`'s
"Evaluation & checkpointing" comments): during training, `eval_dataset` is the fast subsample,
so Trainer's own best-checkpoint tracking would be picking against that subsample, not the full
validation set -- final selection is done manually in step 14, over the full set.

In [ ]:
from trl import SFTConfig, SFTTrainer

def build_trainer(output_dir, eval_dataset, max_steps=None, num_train_epochs=None, eval_steps=None, logging_steps=None):
    sft_config = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=config["per_device_train_batch_size"],
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        learning_rate=float(config["learning_rate"]),
        lr_scheduler_type=config["lr_scheduler_type"],
        warmup_ratio=config["warmup_ratio"],
        weight_decay=config["weight_decay"],
        optim=config["optim"],
        max_grad_norm=config["max_grad_norm"],
        num_train_epochs=num_train_epochs if num_train_epochs is not None else config["num_train_epochs"],
        max_steps=max_steps if max_steps is not None else -1,
        eval_strategy=config["eval_strategy"],
        eval_steps=eval_steps if eval_steps is not None else config["eval_steps"],
        save_strategy=config["save_strategy"],
        save_steps=eval_steps if eval_steps is not None else config["save_steps"],
        logging_steps=logging_steps if logging_steps is not None else config["logging_steps"],
        save_total_limit=config["save_total_limit"],
        load_best_model_at_end=config["load_best_model_at_end"],  # always False -- see cell 9's note
        metric_for_best_model=config["metric_for_best_model"],
        greater_is_better=config["greater_is_better"],
        max_seq_length=config["max_seq_len"],
        seed=config["seed"],
        bf16=USE_BF16,   # decided at runtime in cell 1 (GPU check) -- False on Turing (e.g. T4)
        fp16=not USE_BF16,
        report_to=[],
        push_to_hub=False,  # pushed manually later, after training finishes and the best checkpoint is loaded
    )
    return SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=raw_datasets["train"],
        eval_dataset=eval_dataset,
        data_collator=collator,
        processing_class=tokenizer,
    )

## 11. Smoke test (~100 steps)

Confirms the loss is actually falling and the model produces valid JSON **before** committing
to the full multi-epoch run. Uses `smoke_test.*` from `train/config.yaml`, evaluated against the
subsample (speed only, same as the full run below).

In [ ]:
smoke_output_dir = f"{REPO_ROOT}/{config['smoke_test_output_dir']}"
smoke_trainer = build_trainer(
    output_dir=smoke_output_dir,
    eval_dataset=eval_subsample,
    max_steps=config["smoke_test"]["max_steps"],
    eval_steps=config["smoke_test"]["eval_steps"],
    logging_steps=config["smoke_test"]["logging_steps"],
)
smoke_result = smoke_trainer.train()
print(smoke_result)

# --- Checkpoint disk-size check (LoRA adapters only, not full model weights) ------------------
# save_total_limit is null (train/config.yaml) -- EVERY checkpoint from the full run is kept,
# on purpose (see train/config.yaml's comment: with load_best_model_at_end=False, a finite
# limit would silently delete the true best checkpoint before step 14 gets a chance to find
# it). That only works if Colab's disk can actually hold all of them -- checked HERE, before
# the full run, using a real saved checkpoint from this smoke test, not a guess.
import glob, math, shutil
from pathlib import Path

def dir_size_bytes(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())

smoke_ckpt_dirs = sorted(
    glob.glob(f"{smoke_output_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1])
)
assert smoke_ckpt_dirs, f"No checkpoint-* directories found under {smoke_output_dir} -- smoke test didn't save any."
one_ckpt_dir = smoke_ckpt_dirs[-1]
one_ckpt_mb = dir_size_bytes(one_ckpt_dir) / (1024 ** 2)
print(f"\nOne saved checkpoint ({one_ckpt_dir}): {one_ckpt_mb:.1f} MB on disk "
      "(LoRA adapter only -- the 4-bit base model is never re-saved).")

# Projected checkpoint count for the FULL run -- same math as train/config.yaml's comment.
effective_batch_size = config["per_device_train_batch_size"] * config["gradient_accumulation_steps"]
steps_per_epoch = math.ceil(len(raw_datasets["train"]) / effective_batch_size)
full_total_steps = steps_per_epoch * config["num_train_epochs"]
full_num_checkpoints = math.ceil(full_total_steps / config["save_steps"])
projected_total_gb = (one_ckpt_mb * full_num_checkpoints) / 1024
print(f"Full run: {full_total_steps} steps ({steps_per_epoch}/epoch x {config['num_train_epochs']} epochs), "
      f"save_steps={config['save_steps']} -> ~{full_num_checkpoints} checkpoints "
      f"(save_total_limit is null, so all of them survive).")
print(f"Projected total disk for ALL full-run checkpoints: ~{projected_total_gb:.2f} GB")

free_gb = shutil.disk_usage(REPO_ROOT).free / (1024 ** 3)
print(f"Free disk space at REPO_ROOT ({REPO_ROOT}): {free_gb:.2f} GB")

if projected_total_gb > free_gb:
    print(
        "\n*** WARNING: projected checkpoint total EXCEEDS free disk space at REPO_ROOT. ***\n"
        "Do NOT proceed to the full training run as configured -- this is a real problem, not "
        "a formality. Report this back before continuing so a fix (a smaller save_total_limit, "
        "a different REPO_ROOT/output_dir, clearing space, etc.) can be decided together -- "
        "this notebook will NOT silently reduce save_total_limit on its own."
    )
else:
    print("Free disk space looks sufficient for all full-run checkpoints (LoRA adapters only).")

### Smoke-test loss curve

In [ ]:
import matplotlib.pyplot as plt

history = smoke_trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss", marker="o")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label=f"eval loss ({len(eval_subsample)}-example subsample)", marker="s")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Smoke test loss (should trend downward)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 5 generated outputs on val examples (smoke-test model)

In [ ]:
import json
from ticket_router.schema import TicketLabels
from pydantic import ValidationError

def generate_one(model, messages, max_new_tokens=64):
    prompt_ids = tokenizer.apply_chat_template(
        messages[:2], tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            prompt_ids, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    completion_ids = out_ids[0][prompt_ids.shape[1]:]
    return tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

n_show = config["smoke_test"]["n_generated_examples"]
for i in range(n_show):
    example = raw_datasets["val"][i]
    generated = generate_one(smoke_trainer.model, example["messages"])
    gold = example["messages"][2]["content"]
    try:
        TicketLabels(**json.loads(generated))
        status = "VALID JSON, matches schema"
    except (json.JSONDecodeError, ValidationError) as e:
        status = f"INVALID: {e}"
    print(f"--- val[{i}] ---")
    print(f"  generated: {generated!r}")
    print(f"  gold:      {gold!r}")
    print(f"  status:    {status}\n")

## 12. STOP AND CONFIRM before continuing

**Do not run the cells below until you've checked, above:**
- The smoke-test loss curve is trending **downward** (not flat, not increasing).
- At least most of the 5 generated outputs are **valid JSON matching the 3-key schema**
  (`queue`, `priority`, `type`) -- they do not need to be *correct* yet at only ~100 steps,
  just structurally valid.
- The checkpoint disk-size projection printed at the end of the smoke-test cell did **not**
  print a `WARNING`. If it did, stop here -- `save_total_limit` is null so every full-run
  checkpoint is kept, and this is where a real disk shortfall would be caught before it
  happens partway through the full run.

If any of these look wrong, stop here and investigate (check `train/config.yaml`, the masking
cell's output, and the collator/model setup above) before spending a full ~45-90 minute run
on it. Everything below this cell is the **full, multi-epoch training run**.

## 13. Full training run

Mid-training evaluation (every `eval_steps`=250) runs against the `eval_subsample` -- speed
only. The full validation set is used once, in step 14 below, to pick the actual best
checkpoint.

In [ ]:
full_output_dir = f"{REPO_ROOT}/{config['output_dir']}"
trainer = build_trainer(output_dir=full_output_dir, eval_dataset=eval_subsample)
train_result = trainer.train()
print(train_result)

### Full-run loss curve

In [ ]:
history = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_pts:
    xs, ys = zip(*train_pts)
    plt.plot(xs, ys, label="train loss")
if eval_pts:
    xs, ys = zip(*eval_pts)
    plt.plot(xs, ys, label=f"eval loss ({len(eval_subsample)}-example subsample)", marker="s")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Full run: train loss vs. SUBSAMPLE validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(
    f"NOTE: the eval loss above is against the {len(eval_subsample)}-example subsample (speed "
    "only) -- it is NOT the full-validation metric and is NOT used to pick the final checkpoint. "
    "load_best_model_at_end is False for exactly this reason; the next cell selects the best "
    f"checkpoint using all {len(raw_datasets['val'])} full validation examples instead."
)
print(f"Checkpoints saved during this run (save_steps={config['save_steps']}, "
      f"save_total_limit={config['save_total_limit']}) are on disk under: {full_output_dir}")

## 14. Select the best checkpoint using the FULL validation set

Per `train/config.yaml`: mid-training evaluation used the `eval_subsample` for speed, so it
cannot tell us which of the checkpoints saved during training is actually best. `save_total_limit`
is null (see `train/config.yaml`), so every checkpoint written during the full run is still on
disk -- none were evicted by Trainer's rotation, which is exactly why this step can look at all
of them instead of only whichever ones happened to survive. This cell re-evaluates every
checkpoint saved under `output_dir` on the **full** 1,635-example validation set, by
`metric_for_best_model` (`eval_loss`, `greater_is_better`=False), and loads the winner into
`trainer.model` so every cell below (save, push, preview) uses it.

**Not executable/testable locally** (no GPU here -- same constraint noted at the top of this
notebook and in `PROGRESS.md`'s step-07 entry). Written to the current standard
`transformers`/`trl`/`peft` checkpoint-loading API; report back if anything needs correcting.

In [ ]:
import glob
from peft import PeftModel

def evaluate_checkpoint_on_full_val(checkpoint_dir):
    """Load one saved LoRA checkpoint fresh and evaluate it on the FULL validation set."""
    base = AutoModelForCausalLM.from_pretrained(
        config["base_model"], quantization_config=bnb_config, device_map="auto",
    )
    ckpt_model = PeftModel.from_pretrained(base, checkpoint_dir)
    eval_only_config = SFTConfig(
        output_dir=f"{checkpoint_dir}/_full_val_eval_tmp",
        per_device_eval_batch_size=config["per_device_eval_batch_size"],
        max_seq_length=config["max_seq_len"],
        seed=config["seed"],
        bf16=USE_BF16,
        fp16=not USE_BF16,
        report_to=[],
    )
    eval_trainer = SFTTrainer(
        model=ckpt_model,
        args=eval_only_config,
        eval_dataset=raw_datasets["val"],  # the FULL validation set, on purpose
        data_collator=collator,
        processing_class=tokenizer,
    )
    metrics = eval_trainer.evaluate()
    del base, ckpt_model, eval_trainer
    torch.cuda.empty_cache()
    return metrics["eval_loss"]

checkpoint_dirs = sorted(
    glob.glob(f"{full_output_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1])
)
assert checkpoint_dirs, f"No checkpoint-* directories found under {full_output_dir} -- did training actually save any?"
print(f"Found {len(checkpoint_dirs)} saved checkpoint(s): {checkpoint_dirs}")

full_val_losses = {}
for ckpt_dir in checkpoint_dirs:
    loss = evaluate_checkpoint_on_full_val(ckpt_dir)
    full_val_losses[ckpt_dir] = loss
    print(f"  {ckpt_dir}: FULL-val eval_loss = {loss:.4f}  (n={len(raw_datasets['val'])})")

best_ckpt_dir = min(full_val_losses, key=full_val_losses.get)
print(f"\nBest checkpoint by FULL-val eval_loss: {best_ckpt_dir}  (eval_loss={full_val_losses[best_ckpt_dir]:.4f})")

# Load the winner into `trainer.model` -- every cell below (save/push/preview) now uses the
# checkpoint selected on the FULL validation set, not the subsample-monitored last step.
best_base = AutoModelForCausalLM.from_pretrained(
    config["base_model"], quantization_config=bnb_config, device_map="auto",
)
trainer.model = PeftModel.from_pretrained(best_base, best_ckpt_dir)
print(f"trainer.model now holds the checkpoint from {best_ckpt_dir}.")

## 15. Save the adapter + push to the Hugging Face Hub

`trainer.model` is the checkpoint selected in step 14 by FULL-validation `eval_loss`. Requires
`hub_model_id` in `train/config.yaml` (already set to
`Omar-Mohsen22/ticket-router-qwen2.5-1.5b-qlora`) and a write-access HF token, already confirmed
present in step 5.

In [ ]:
adapter_dir = f"{REPO_ROOT}/{config['adapter_output_dir']}"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter + tokenizer saved to {adapter_dir}")

if config["push_to_hub"]:
    if not config["hub_model_id"]:
        raise ValueError(
            "push_to_hub is true in train/config.yaml but hub_model_id is empty. "
            "Set hub_model_id to '<your-hf-username>/<repo-name>' and re-run this cell."
        )
    # Reuses the token already validated (write access confirmed) in step 5 -- no interactive
    # login() prompt needed here.
    trainer.model.push_to_hub(config["hub_model_id"], private=config["hub_private"], token=token)
    tokenizer.push_to_hub(config["hub_model_id"], private=config["hub_private"], token=token)
    print(f"Pushed adapter to https://huggingface.co/{config['hub_model_id']}")
else:
    print("push_to_hub is false in train/config.yaml -- adapter saved locally only.")

## 16. Post-train preview: 20 val examples

A quick eyeball check of the trained model's output format and rough quality, right after
training and before running the project's full evaluation harness (`evaluate.py`) separately.
Uses `post_train_preview.n_val_examples` from `train/config.yaml`.

In [ ]:
n_preview = config["post_train_preview"]["n_val_examples"]
preview_rows = []
for i in range(n_preview):
    example = raw_datasets["val"][i]
    generated = generate_one(trainer.model, example["messages"])
    gold = example["messages"][2]["content"]
    try:
        parsed = TicketLabels(**json.loads(generated))
        status = "valid"
    except (json.JSONDecodeError, ValidationError) as e:
        parsed, status = None, f"invalid: {e}"
    preview_rows.append((i, generated, gold, status))
    print(f"--- val[{i}] ({status}) ---")
    print(f"  generated: {generated!r}")
    print(f"  gold:      {gold!r}\n")

n_valid = sum(1 for _, _, _, s in preview_rows if s == "valid")
n_exact = sum(1 for _, g, gold, s in preview_rows if s == "valid" and g == gold)
print(f"{n_valid}/{n_preview} valid JSON, {n_exact}/{n_preview} exact string match on these {n_preview} examples.")
print("This is a quick eyeball check only -- run src/ticket_router/evaluate.py locally against "
      "a full predictions file for real macro-F1/accuracy numbers on the harness.")